# Problem 57: Multi-Index RAG

**Difficulty:** Hard | **Framework:** LangChain

**Categories:** rag

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/multi-index-rag).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- FAQ and technical documentation must be stored in separate vector store indexes
- The agent must route queries to the correct index based on query type
- The routing decision must happen before retrieval, not after
- Both indexes must remain searchable for their respective query types


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

faq_docs = [
    Document(page_content="Q: How do I reset my password? A: Go to Settings > Security > Reset Password."),
    Document(page_content="Q: What payment methods do you accept? A: We accept Visa, Mastercard, and PayPal."),
    Document(page_content="Q: How do I cancel my subscription? A: Go to Billing > Manage Subscription > Cancel."),
]

technical_docs = [
    Document(page_content="POST /api/v2/auth/reset-password - Requires email field. Returns 200 with reset token."),
    Document(page_content="GET /api/v2/billing/subscription - Returns subscription object with status, plan, and next_billing_date."),
    Document(page_content="Authentication: All API requests require Bearer token in Authorization header."),
]

# BUG: Everything dumped into a single index — FAQ and technical docs get mixed
all_docs = faq_docs + technical_docs
vectorstore = FAISS.from_documents(all_docs, embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on context.\n\nContext: {context}"),
    ("human", "{question}"),
])

def ask(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    chain = prompt | llm
    result = chain.invoke({"context": context, "question": question})
    return result.content

# FAQ question gets technical API docs mixed in
print(ask("How do I reset my password?"))
# Technical question gets FAQ mixed in
print(ask("What is the API endpoint for password reset?"))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Classify the query type (FAQ vs technical) before choosing which index to search
2. Use an LLM or rule-based classifier to determine the appropriate index
3. Consider that some queries may need results from both indexes


## 7. Evaluation Criteria

Check your solution against these criteria:

- Uses separate indexes
- Routes queries to correct index
- Returns accurate answers
